![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [ ]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [ ]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

In [ ]:
print("Missing values:")
print(tables["trips"].isna().sum())

print("\nTrip duration summary:")
print(tables["trips"]["duration_min"].describe())

print("\nBike types:")
print(tables["bikes"]["bike_type"].value_counts(dropna=False))

Missing values:
trip_id              0
bike_id              0
start_station_id     0
end_station_id       0
start_time           0
duration_min        30
user_type            0
dtype: int64

Trip duration summary:
count     582.000000
mean       27.891753
std       145.064305
min         4.000000
25%        12.000000
50%        16.000000
75%        22.000000
max      2880.000000
Name: duration_min, dtype: float64

Bike types:
bike_type
Standard     36
Electric      9
STANDARD      8
standard      3
Electric      2
electric      1
E-bike        1
Name: count, dtype: int64


### 1. Look at the Data

**What does one row of each table represent?**

- **Trips:** One recorded bike journey, including the bike used, start and end stations, and duration.
- **Bikes:** One individual bike, including its type and purchase year.
- **Stations:** One bike station, including its name, zone and capacity.

**Which columns will I use?**

- **Trips:** `trip_id`, `bike_id`, `start_station_id`, `duration_min`
- **Bikes:** `bike_id`, `bike_type`
- **Stations:** `station_id`, `zone`

These columns are needed to calculate the number of trips and average trip duration by bike type and starting-station zone.

**What looks odd, missing or inconsistent?**

1. **Repeated trip IDs:** The trips table contains 612 rows but only 600 unique trip IDs, suggesting duplicate records that need investigation.

2. **Missing values:** There are 30 missing values in `duration_min`. These could affect the average trip duration calculation.

3. **Unusual durations:** The minimum duration is 4 minutes, but the maximum is 2,880 minutes (48 hours). This seems unusually long for a bike journey and should be investigated before calculating averages.

4. **Inconsistent bike types:** Bike types are written differently, including `Standard`, `STANDARD`, `standard`, `Electric`, `electric` and `E-bike`. These should be standardised so the same bike type is not treated as multiple categories.

5. **Inconsistent user types:** The data contains both `Casual` and `CASUAL`, showing inconsistent capitalisation.


## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

In [ ]:
trips_clean = tables["trips"].copy()
bikes_clean = tables["bikes"].copy()
stations_clean = tables["stations"].copy()

In [ ]:
print("Exact duplicate rows:", trips_clean.duplicated().sum())

print("Repeated trip IDs:",
      trips_clean.duplicated(subset=["trip_id"]).sum())

Exact duplicate rows: 12
Repeated trip IDs: 12


In [ ]:
trips_clean = trips_clean.drop_duplicates()

print("Rows after cleaning:", len(trips_clean))
print("Remaining duplicates:", trips_clean.duplicated().sum())

Rows after cleaning: 600
Remaining duplicates: 0


I cleaned the data before joining the tables to ensure the analysis would be reliable. I used separate code cells to make the process easy to check and repeat.

**Cell 1 – Creating copies of the datasets**

I created copies of the trips, bikes and stations tables using `.copy()`. This ensures the original data remains unchanged while I clean and prepare the datasets.

**Cell 2 – Identifying duplicates**

I used `.duplicated().sum()` to check for duplicate records. I found 12 exact duplicate rows and 12 repeated trip IDs, confirming that the repeated IDs were accounted for by exact duplicates.

**Cell 3 – Removing duplicates**

I used `.drop_duplicates()` to remove the 12 duplicate trips, reducing the dataset from 612 to 600 records. This prevents the same journeys being counted more than once.

**Other data-quality decisions**

- **Missing durations:** I identified 30 missing values in `duration_min`. I excluded these from the duration analysis rather than estimating values, as this could distort the average.
- **Unusual durations:** The maximum duration was 2,880 minutes (48 hours). I investigated this as a potential outlier rather than automatically deleting it, because it could represent a genuine rental.
- **Inconsistent bike types:** I standardised variations such as `STANDARD`, `standard` and `Standard`, as well as `Electric`, `electric` and `E-bike`, into consistent categories. This ensures accurate grouping.
- **Inconsistent user types:** I standardised differences such as `Casual` and `CASUAL` for consistency.

**Validation and reasoning**

My main aim was to prevent duplicate records, missing durations and inconsistent categories from affecting the final average trip duration. I kept the original data unchanged and used separate, repeatable cleaning steps.

I used ChatGPT to help understand the cleaning methods and decisions, and checked the outputs in Colab.

## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

In [ ]:
print("Trips before join:", len(trips_clean))

joined = trips_clean.merge(
    bikes_clean,
    on="bike_id",
    how="left",
    validate="many_to_one"
)

print("Rows after bike join:", len(joined))
print("Unmatched bikes:", joined["bike_type"].isna().sum())

Trips before join: 600
Rows after bike join: 600
Unmatched bikes: 0


In [ ]:
joined = joined.merge(
    stations_clean[["station_id", "zone"]],
    left_on="start_station_id",
    right_on="station_id",
    how="left",
    validate="many_to_one"
)

print("Rows after station join:", len(joined))
print("Unmatched stations:", joined["zone"].isna().sum())

display(joined[["trip_id", "bike_type", "zone", "duration_min"]].head())

Rows after station join: 600
Unmatched stations: 0


,trip_id,bike_type,zone,duration_min
0,T0001,Standard,South,21.0
1,T0002,standard,Centre,27.0
2,T0003,Standard,West,18.0
3,T0004,Electric,West,34.0
4,T0005,Standard,Centre,16.0


In [ ]:
joined["bike_type"] = (
    joined["bike_type"]
    .str.strip()
    .str.lower()
    .replace({"e-bike": "electric"})
    .str.title()
)

print(joined["bike_type"].value_counts())

bike_type
Standard    426
Electric    174
Name: count, dtype: int64


In [ ]:
result = (
    joined.groupby(["bike_type", "zone"])
    .agg(
        trip_count=("duration_min", "count"),
        avg_duration_min=("duration_min", "mean")
    )
    .reset_index()
)

result["avg_duration_min"] = result["avg_duration_min"].round(2)

display(result)

,bike_type,zone,trip_count,avg_duration_min
0,Electric,Centre,81,14.51
1,Electric,East,11,15.73
2,Electric,North,23,15.96
3,Electric,South,24,13.42
4,Electric,West,22,15.68
5,Standard,Centre,217,38.66
6,Standard,East,20,19.50
7,Standard,North,39,20.36
8,Standard,South,60,43.15
9,Standard,West,73,19.82




**Cell 1 – Join trips and bikes**

I used `.merge()` on `bike_id` with a left join and `validate="many_to_one"`.

- Expected: 600 rows
- Actual: 600 rows
- Unmatched bikes: 0

**Cell 2 – Join starting stations**

I joined `start_station_id` to `station_id` to obtain the starting zone.

- Expected: 600 rows
- Actual: 600 rows
- Unmatched stations: 0

Both joins preserved the row count without creating duplicates.

**Cell 3 – Standardise bike types**

I standardised inconsistent labels using `.str.lower()`, `.replace()` and `.str.title()`, ensuring electric and standard bikes were grouped correctly.

**Cell 4 – Aggregate**

I used `groupby(["bike_type", "zone"])`, `count()` and `mean()` to calculate trip counts and average durations.

- Final output: 10 groups
- Trips with recorded durations: 570
- Missing durations excluded from averages: 30

**Findings:** Standard bikes had longer average durations in all five zones. However, unusually long trips may distort the averages and require further investigation.

## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

In [ ]:
print("Total trips:", len(joined))
print("Trips with durations:", joined["duration_min"].count())
print("Grouped trip count:", result["trip_count"].sum())

Total trips: 600
Trips with durations: 570
Grouped trip count: 570


In [ ]:
check = joined[
    (joined["bike_type"] == "Electric") &
    (joined["zone"] == "Centre")
]

print("Trips:", check["duration_min"].count())
print("Average:", round(check["duration_min"].mean(), 2))

Trips: 81
Average: 14.51


In [ ]:
display(
    joined[joined["duration_min"] > 180][
        ["trip_id", "bike_type", "zone", "duration_min"]
    ]
)

,trip_id,bike_type,zone,duration_min
80,T0081,Standard,Centre,2880.0
142,T0143,Standard,Centre,1440.0
283,T0284,Standard,South,1440.0




**Cell 1 – Count validation**

I confirmed that there were 600 trips, of which 570 had recorded durations. The grouped trip counts also totalled 570, confirming consistency.

**Cell 2 – Independent calculation**

I recalculated the Electric–Centre group separately and obtained 81 trips with an average duration of 14.51 minutes, matching the final table.

**Cell 3 – Outlier check**

I identified three unusually long trips:
- T0081: 2,880 minutes (Standard, Centre)
- T0143: 1,440 minutes (Standard, Centre)
- T0284: 1,440 minutes (Standard, South)

**Conclusion:** The counts and spot-check matched, supporting the accuracy of the aggregation. However, these three extreme durations may inflate the averages for standard bikes in Centre and South. I would investigate them before drawing firm conclusions.

## 5. Prompts and AI notes





**AI tool:** ChatGPT

### Task 1 – Look at the Data

**Prompts sent:**
1. "lets go step by step"
2. "im confused and a bit lost"
3. "the fuirst taks is look at the dats. and i ahev tow rite it [...] i dotnt hink this part needs coding"
4. "wait for look at data lets als;o find missing or unsual data"
5. "using this lets finish up anwer to 1"

**AI response:** Explained the task, suggested code for checking missing values and unusual data, and helped draft the findings.

**My checks:** Ran the code in Colab and reviewed the actual results before writing Task 1.

### Task 2 – Clean and Prepare

**Prompts sent:**
1. "lets asnwwr q2 qitrh text now sicne all the codes are ran"
2. "lets be mroe clear and link to te coides used too"
3. "i alreayd used the code so lets just refer to vcell 1-3. rathet than images. and lets write the anwer now"

**AI response:** Suggested using `.copy()`, `.duplicated()` and `.drop_duplicates()`, along with approaches for missing values, outliers and inconsistent categories.

**My checks:** Confirmed 12 exact duplicates and 12 repeated trip IDs. The dataset contained 600 rows after duplicate removal

## 6. What I could not verify

*What remains unchecked? What would you do next?*



- **Outliers:** I identified three unusually long trips (1,440–2,880 minutes) but could not confirm whether they were genuine rentals or data-entry errors.
- **Missing durations:** I could not determine why 30 trips had missing durations or whether excluding them could bias the results.
- **Data accuracy:** I could not verify the original trip records against the source system.

**Next steps:** Investigate the unusual trips and missing values, compare averages with and without outliers, and verify questionable records against the original data before making recommendations.